# 신용카드 고객 데이터 기반 `income_id` 추정 및 미확인 고객군 분류 분석 보고서

## 보고서 목적
본 노트북은 제공된 파이썬 코드를 기반으로, `income_id`가 알려진 고객 데이터를 활용해 **소득 구간을 구분하는 기준을 탐색**하고,  
`income_id == 6`으로 표기된 **미확인 소득군(Unknown)** 에 대해 모델 기반으로 분류 가능성을 검토하기 위한 실무형 분석 보고서입니다.

## 분석 시나리오
- 전체 고객 데이터를 불러온다.
- `income_id`를 타깃으로 보기 위한 기본 품질 점검을 수행한다.
- `income_id == 6`을 미확인 소득군으로 분리한다.
- 알려진 소득군을 기준으로 LDA 시각화를 수행해 군집 분리 가능성을 확인한다.
- 실무 적용을 고려해 `income_id <= 2`와 `income_id >= 3`을 이진 분류로 재정의한다.
- Logistic Regression, RandomForestClassifier를 비교하여 분류 성능과 해석 가능성을 함께 검토한다.
- 마지막으로 Unknown 그룹에 대한 예측 결과를 비교하여 운영 관점의 활용 가능성을 점검한다.

## 분석 포인트
- `income_id`의 원래 다중 클래스 구조가 실제로 어느 정도 분리 가능한지 확인하는 것이 중요하다.
- Unknown 그룹을 곧바로 학습에 포함하면 라벨 오염 가능성이 있으므로, 먼저 Known/Unknown 분리가 필요하다.
- 실무에서는 다중 클래스보다 **업무 의사결정이 가능한 이진 기준**이 더 유용한 경우가 많다.
- 모델 간 예측 불일치 구간은 운영 배포 전 추가 검증 대상이 된다.

## 1. 분석 환경 설정

### 비즈니스 목적
분석에 필요한 기본 라이브러리를 먼저 불러와, 이후 셀들이 동일한 환경에서 안정적으로 재현되도록 준비합니다.

### 기대 효과
- 재실행 시 환경 차이로 인한 오류를 줄일 수 있습니다.
- 팀 단위 협업 시 어떤 패키지가 필요한지 빠르게 파악할 수 있습니다.

### 분석 포인트
- 시각화, 데이터 처리, API 호출용 라이브러리를 분리해 관리합니다.
- 노트북 상단에서 공통 의존성을 선언하면 유지보수성이 좋아집니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from IPython.display import display

# 왜 이 설정을 하는가:
# - 노트북 실행 시 경고/표시 형식을 일정하게 맞춰야 협업자가 결과를 동일하게 해석할 수 있기 때문입니다.
# - display() 기반 출력은 print()보다 표 형태 데이터 확인에 유리하므로 실무 보고서 가독성이 좋아집니다.
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 배포 및 유지보수 주의:
# - 실행 환경에 matplotlib GUI 설정 차이가 있으면 일부 IDE에서 그래프 출력 방식이 다를 수 있습니다.
# - pandas 옵션은 세션 전체에 적용되므로, 다른 노트북과 함께 실행할 경우 출력 형식이 달라질 수 있습니다.

## 2. 데이터 조회 함수 정의

### 비즈니스 목적
분석 대상 데이터를 외부 파이프라인 API로부터 불러오기 위한 표준 함수를 정의합니다.

### 기대 효과
- 데이터 적재 로직을 함수화하면 재사용성과 테스트 용이성이 높아집니다.
- 추후 API 경로 변경 시 함수 내부만 수정하면 되어 유지보수가 쉬워집니다.

### 분석 포인트
- 현재 함수는 전체 데이터 또는 X/y 분리 형태를 모두 지원합니다.
- API 응답 구조가 바뀌면 downstream 셀 전체가 영향을 받으므로 초기에 구조를 명확히 확인해야 합니다.

In [ ]:
def fetch_creditcard(X_y_split: bool = False) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    # 왜 함수로 분리하는가:
    # - 데이터 소스 접근 로직을 한 곳에 모아두면, URL/파라미터/응답 구조 변경에 빠르게 대응할 수 있기 때문입니다.
    # - 재현 가능한 분석에서는 '같은 방식으로 데이터를 읽는다'는 보장이 매우 중요합니다.
    response = requests.get(
        "http://pipeline:8000/dataset/creditcard-churn",
        params={"X_y_split": X_y_split},
        timeout=30
    )
    response.raise_for_status()
    payload = response.json()

    if X_y_split:
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"], name="target")
        return X, y

    df = pd.DataFrame(payload["data"], index=payload["index"])
    return df

# 배포 및 유지보수 주의:
# - 사내 환경이 아닌 곳에서는 'http://pipeline:8000' 경로가 동작하지 않을 수 있습니다.
# - API 응답 키 이름(x, y, data, index)이 바뀌면 즉시 에러가 발생하므로 스키마 관리가 필요합니다.
# - timeout을 지정하지 않으면 네트워크 이상 시 노트북이 장시간 멈춘 것처럼 보일 수 있습니다.

## 3. 원천 데이터 로드 및 1차 확인

### 비즈니스 목적
실제 분석 대상 데이터를 적재하고, 불필요한 식별자 컬럼을 제거한 뒤 기본 행 수준 구조를 확인합니다.

### 기대 효과
- 모델링에 필요 없는 ID 컬럼을 제거해 데이터 누수 가능성을 줄일 수 있습니다.
- 컬럼 목록과 샘플 데이터를 먼저 확인하면 이후 전처리 방향을 빨리 결정할 수 있습니다.

### 분석 포인트
- `creditcard_churn_id`는 식별자 역할이므로 학습 변수에서 제외하는 것이 적절합니다.
- 샘플 5행과 컬럼 목록을 함께 보면 범주형/수치형 혼재 여부를 빠르게 파악할 수 있습니다.

In [ ]:
# 왜 먼저 원천 데이터를 그대로 불러오는가:
# - 전처리 이전 원본 상태를 확인해야, 이후 단계에서 어떤 컬럼이 제거/변환되었는지 추적할 수 있기 때문입니다.
bank_df = fetch_creditcard()

# 왜 식별자 컬럼을 제거하는가:
# - 식별자 값은 비즈니스 의미보다 고유성만 가지므로, 모델이 패턴이 아닌 레코드 자체를 외우는 데이터 누수 위험이 있습니다.
if "creditcard_churn_id" in bank_df.columns:
    bank_df = bank_df.drop("creditcard_churn_id", axis=1)

display(bank_df.head())
display(pd.DataFrame({"column_name": bank_df.columns}))

# 배포 및 유지보수 주의:
# - 운영 데이터셋에 컬럼이 추가/삭제될 수 있으므로, drop 전에 존재 여부를 확인하는 방식이 안전합니다.
# - API 단계에서 dtype이 예상과 다르게 들어오면 후속 통계/모델링 셀에서 에러가 날 수 있습니다.

## 4. 데이터 구조 및 기초 품질 점검

### 비즈니스 목적
전체 데이터의 행·열 크기, 결측치, 데이터 타입을 먼저 점검하여 전처리 리스크를 사전에 확인합니다.

### 기대 효과
- 결측치나 비정상 dtype으로 인한 모델 학습 오류를 미리 방지할 수 있습니다.
- 이후 통계 분석, 스케일링, 시각화 단계의 안정성을 높일 수 있습니다.

### 분석 포인트
- 숫자처럼 보이지만 문자열로 들어온 컬럼이 있는지 확인해야 합니다.
- 결측치가 없다면 현재 데이터셋은 비교적 바로 모델링하기 쉬운 구조일 가능성이 높습니다.

In [ ]:
summary_df = pd.DataFrame({
    "dtype": bank_df.dtypes.astype(str),
    "missing_count": bank_df.isna().sum(),
    "missing_ratio": bank_df.isna().mean().round(4)
})

display(pd.DataFrame({
    "row_count": [bank_df.shape[0]],
    "column_count": [bank_df.shape[1]]
}))
display(summary_df)

# 왜 기초 품질 점검을 먼저 하는가:
# - 모델링 전 결측치와 dtype 오류를 잡지 않으면, 학습 단계에서 원인 파악이 어려운 예외가 발생할 수 있기 때문입니다.
# - 특히 API 기반 적재 데이터는 날짜/범주형이 object 타입으로 들어오는 경우가 많아 사전 확인이 필요합니다.

## 5. 타깃(`income_id`)과 입력 변수 분리

### 비즈니스 목적
이번 분석의 목적 변수인 `income_id`를 분리하고, 나머지 컬럼을 입력 변수 후보로 구성합니다.

### 기대 효과
- 타깃과 피처를 분리해 데이터 누수 가능성을 줄일 수 있습니다.
- 이후 분석 흐름을 Known/Unknown, 학습/평가 구조로 명확히 구분할 수 있습니다.

### 분석 포인트
- 현재 원본 코드에서는 `income_id`를 예측 대상으로 보고 있으므로 피처 목록에서 제외합니다.
- 타깃 분포를 먼저 확인해야 클래스 불균형 여부를 판단할 수 있습니다.

In [ ]:
features = [col for col in bank_df.columns if col != "income_id"]
features_df = bank_df[features].copy()
income_df = bank_df["income_id"].copy()

display(features_df.head())
display(income_df.value_counts().sort_index().rename_axis("income_id").to_frame("count"))

# 왜 타깃을 먼저 분리하는가:
# - 목적 변수가 입력 피처에 남아 있으면 학습 성능이 비정상적으로 높아지는 누수 문제가 발생할 수 있기 때문입니다.
# - copy()를 사용하면 후속 가공 과정에서 SettingWithCopyWarning 가능성을 줄일 수 있습니다.
# 배포 및 유지보수 주의:
# - income_id가 정수형 범주라는 전제가 깨지면, 후속 이진 라벨 생성 로직이 잘못 동작할 수 있습니다.

## 6. 전체 피처 기초 통계 확인

### 비즈니스 목적
입력 변수 전반의 분포 범위와 이상치 가능성을 빠르게 파악하여, 스케일링/모델 선택 방향을 정리합니다.

### 기대 효과
- 변수별 값의 범위 차이를 파악해 표준화 필요성을 판단할 수 있습니다.
- 일부 변수의 편향이나 이상치 구간을 조기에 인지할 수 있습니다.

### 분석 포인트
- `credit_limit`, `transaction_amount` 등은 값 범위가 큰 편일 가능성이 높습니다.
- 서로 스케일 차이가 큰 변수들이 섞여 있으면 거리 기반/선형 모델에서 표준화 효과가 큽니다.

In [ ]:
display(features_df.describe().T)

# 왜 describe()를 별도 셀로 분리하는가:
# - 기초 통계는 데이터의 전반적 건강 상태를 보는 첫 단계이며, 이후 그래프와 모델 해석의 기준점이 되기 때문입니다.
# - transpose(.T)하여 보여주면 컬럼별 최소/최대/사분위수를 한 눈에 비교하기 쉽습니다.

## 7. Known / Unknown 소득군 분리

### 비즈니스 목적
`income_id == 6`을 미확인 소득군으로 가정하고, 알려진 라벨과 미확인 라벨을 분리해 분석합니다.

### 기대 효과
- 라벨이 확실한 데이터만으로 모델을 학습하여 신뢰도 높은 패턴을 확보할 수 있습니다.
- Unknown 그룹은 예측 적용 대상으로 별도 관리할 수 있습니다.

### 분석 포인트
- Unknown을 학습에 포함하면 라벨 의미가 불명확해져 모델 품질이 저하될 수 있습니다.
- Known 데이터의 클래스 분포를 다시 확인해야 학습 가능성을 판단할 수 있습니다.

In [ ]:
features_known_df = features_df[income_df != 6].copy()
features_unknown_df = features_df[income_df == 6].copy()

income_known_df = income_df[income_df != 6].copy()
income_unknown_df = income_df[income_df == 6].copy()

display(pd.DataFrame({
    "dataset": ["known", "unknown"],
    "row_count": [len(features_known_df), len(features_unknown_df)]
}))

display(income_known_df.value_counts().sort_index().rename_axis("income_id").to_frame("count"))

# 왜 Unknown을 먼저 분리하는가:
# - 라벨 신뢰도가 낮은 표본을 학습 데이터에 넣으면, 모델이 모호한 기준을 학습해 성능과 해석 가능성이 동시에 떨어질 수 있기 때문입니다.
# - 실무에서는 '학습 데이터의 라벨 품질'이 모델 성능보다 더 중요한 경우가 많습니다.

## 8. Known / Unknown 그룹의 요약 통계 비교

### 비즈니스 목적
Unknown 그룹이 Known 그룹과 얼마나 다른 분포를 가지는지 정량적으로 확인합니다.

### 기대 효과
- Unknown 그룹이 기존 Known 분포 안에 있는지, 혹은 다른 성격의 고객군인지 판단할 수 있습니다.
- 이후 예측 결과 해석 시 '모델 적용 가능 범위'를 점검할 수 있습니다.

### 분석 포인트
- 특정 변수에서 Unknown 그룹 평균/사분위가 Known과 크게 다르면 예측 불확실성이 커질 수 있습니다.
- 분포 차이가 너무 크면 추가 데이터 정의 검토가 필요합니다.

In [ ]:
known_desc = features_known_df.describe().T.add_prefix("known_")
unknown_desc = features_unknown_df.describe().T.add_prefix("unknown_")
comparison_desc = known_desc.join(unknown_desc, how="outer")

display(comparison_desc)

# 왜 Known/Unknown 통계를 따로 비교하는가:
# - 추론 대상 데이터가 학습 데이터와 너무 다른 분포를 가지면, 모델 성능이 운영 환경에서 급격히 저하될 수 있기 때문입니다.
# - 이는 흔히 데이터 드리프트 혹은 적용 범위 이탈 문제를 확인하는 기초 단계입니다.

## 9. 상관관계 분석

### 비즈니스 목적
주요 수치형 변수 간 선형 관계를 확인해, 해석 포인트와 중복 정보 가능성을 점검합니다.

### 기대 효과
- 서로 매우 강한 상관을 가지는 변수 조합을 파악할 수 있습니다.
- 변수 선택, 해석, 모델 단순화 방향을 잡는 데 도움이 됩니다.

### 분석 포인트
- `credit_limit`, `available_credit`, `utilization_ratio`는 구조적으로 관련 가능성이 높습니다.
- 상관관계는 인과관계가 아니므로 해석 시 주의가 필요합니다.

In [ ]:
corr_df = features_df.corr(numeric_only=True)

display(corr_df)

plt.figure(figsize=(12, 9))
plt.imshow(corr_df, aspect="auto")
plt.colorbar(label="상관계수")
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title("수치형 변수 상관관계 히트맵")
plt.xlabel("변수")
plt.ylabel("변수")
plt.tight_layout()
plt.show()

# 왜 상관관계를 보는가:
# - 서로 거의 같은 정보를 담는 변수가 많으면, 해석이 복잡해지고 선형 모델 계수 안정성이 떨어질 수 있기 때문입니다.
# - 특히 실무 보고서에서는 '어떤 변수군이 함께 움직이는지'를 설명하는 데 유용합니다.

## 10. 주요 변수별 Known / Unknown 분포 비교 시각화

### 비즈니스 목적
운영적으로 중요할 가능성이 높은 수치형 변수 몇 개를 골라 Known / Unknown 그룹의 분포 차이를 시각적으로 확인합니다.

### 기대 효과
- 단순 평균 비교보다 실제 분포 형태 차이를 직관적으로 파악할 수 있습니다.
- Unknown 그룹이 특정 영역에 편중되어 있는지 빠르게 발견할 수 있습니다.

### 분석 포인트
- 거래 규모, 이용 한도, 거래 횟수 관련 변수는 소득 구간과 연결될 가능성이 높습니다.
- 박스플롯은 이상치와 중앙값 차이를 동시에 파악하는 데 유리합니다.

In [ ]:
plot_features = [
    "credit_limit",
    "available_credit",
    "transaction_amount",
    "transaction_count",
    "revolving_balance",
    "utilization_ratio"
]

for col in plot_features:
    compare_df = pd.DataFrame({
        "value": pd.concat(
            [
                features_known_df[col].rename("value"),
                features_unknown_df[col].rename("value")
            ],
            axis=0
        ),
        "group": (["Known"] * len(features_known_df)) + (["Unknown"] * len(features_unknown_df))
    })

    plt.figure(figsize=(8, 4))
    plt.boxplot(
        [compare_df.loc[compare_df["group"] == "Known", "value"],
         compare_df.loc[compare_df["group"] == "Unknown", "value"]],
        labels=["Known", "Unknown"]
    )
    plt.title(f"{col} 분포 비교 (Known vs Unknown)")
    plt.xlabel("그룹")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

# 왜 변수별로 세분화해 보는가:
# - 전체 통계만으로는 분포의 꼬리, 이상치, 중앙값 차이를 놓치기 쉽기 때문입니다.
# - 운영 의사결정에서는 '어떤 변수에서 차이가 나는지'가 중요하므로 변수 단위 시각화가 필요합니다.

## 11. 대표 변수에 대한 통계 검정

### 비즈니스 목적
Known / Unknown 그룹 간 차이가 우연인지, 아니면 통계적으로 유의한 차이인지 확인합니다.

### 기대 효과
- 단순 시각적 인상에 의존하지 않고 정량적 근거를 확보할 수 있습니다.
- 후속 보고서에서 '차이가 있다/없다'를 보다 설득력 있게 제시할 수 있습니다.

### 분석 포인트
- 표본 수가 많으면 작은 차이도 유의하게 나올 수 있으므로 효과 크기 해석이 함께 필요합니다.
- 여기서는 실무 간단 검토용으로 t-test를 적용합니다.

In [ ]:
from scipy.stats import ttest_ind

test_results = []

for col in plot_features:
    stat, pvalue = ttest_ind(
        features_known_df[col],
        features_unknown_df[col],
        equal_var=False,
        nan_policy="omit"
    )
    test_results.append({
        "feature": col,
        "t_statistic": stat,
        "p_value": pvalue
    })

test_result_df = pd.DataFrame(test_results).sort_values("p_value")
display(test_result_df)

# 왜 통계 검정을 수행하는가:
# - 그래프에서 차이가 있어 보이더라도, 표본 변동성 때문에 실제로는 우연일 수 있기 때문입니다.
# - 실무에서는 우선순위 선정 목적으로 '차이가 큰 변수 후보'를 정리하는 데 유용합니다.
# 배포 및 유지보수 주의:
# - 매우 비대칭 분포에서는 t-test 가정이 완벽하지 않을 수 있으므로, 필요 시 비모수 검정으로 보완해야 합니다.

## 12. LDA 적용 전 표준화

### 비즈니스 목적
LDA 및 일부 선형 모델에서 변수 스케일 차이의 영향을 줄이기 위해 Known / Unknown 데이터를 동일 기준으로 표준화합니다.

### 기대 효과
- 큰 값 범위를 가진 변수에 과도하게 끌리는 현상을 줄일 수 있습니다.
- LDA 축 해석과 후속 로지스틱 회귀 학습의 안정성이 높아집니다.

### 분석 포인트
- 스케일러는 반드시 Known 데이터에만 fit 하고 Unknown에는 transform만 적용해야 합니다.
- 이는 추론 시점의 데이터 누수를 막는 기본 원칙입니다.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 왜 Known 데이터에만 fit 하는가:
# - 스케일링 기준을 학습 데이터에서만 추정해야, 미확인 데이터 정보가 학습 과정에 섞이는 데이터 누수를 막을 수 있기 때문입니다.
features_known_df_scaled = scaler.fit_transform(features_known_df)
features_unknown_df_scaled = scaler.transform(features_unknown_df)

display(pd.DataFrame(features_known_df_scaled, columns=features_known_df.columns).head())

# 배포 및 유지보수 주의:
# - 운영 배포 시에는 반드시 학습 때 저장한 scaler 객체를 재사용해야 합니다.
# - 새로 fit하면 기준 평균/표준편차가 바뀌어 예측 결과가 달라집니다.

## 13. LDA 기반 2차원 축소

### 비즈니스 목적
Known 소득군이 저차원 공간에서도 어느 정도 분리되는지 확인하고, Unknown 그룹이 어떤 클래스 영역에 가까운지 탐색합니다.

### 기대 효과
- 모델링 전에 분류 가능성에 대한 직관을 얻을 수 있습니다.
- Unknown 그룹의 대략적 위치를 시각적으로 확인할 수 있습니다.

### 분석 포인트
- LDA는 라벨 정보를 사용해 분리 축을 찾으므로, 단순 PCA보다 클래스 분리 확인에 적합합니다.
- 시각화가 잘 분리되지 않아도 실제 예측은 가능할 수 있으므로 참고 지표로 활용합니다.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=2)
features_known_df_lda = lda.fit_transform(features_known_df_scaled, income_known_df)
features_unknown_df_lda = lda.transform(features_unknown_df_scaled)

lda_known_plot_df = pd.DataFrame(features_known_df_lda, columns=["LD1", "LD2"], index=income_known_df.index)
lda_known_plot_df["income_id"] = income_known_df.values

display(lda_known_plot_df.head())

# 왜 LDA를 사용하는가:
# - 이번 목적은 차원 축소 자체보다 '소득 클래스 구분 가능성' 확인이므로, 라벨 정보를 활용하는 LDA가 적합합니다.
# - Unknown 데이터는 transform만 수행하여 기존 분리 축 위에 위치를 비교합니다.

## 14. LDA 다중 시각화

### 비즈니스 목적
개별 클래스 및 클래스 묶음별로 Known 데이터의 분포를 살펴보고, Unknown 그룹의 상대적 위치를 비교합니다.

### 기대 효과
- 어떤 소득 구간끼리 경계가 유사한지 확인할 수 있습니다.
- 이후 이진 분류 재정의의 타당성을 보조적으로 검토할 수 있습니다.

### 분석 포인트
- 클래스 간 중첩이 심하면 다중 클래스 직접 분류보다 구간 통합 전략이 실무적으로 더 유리할 수 있습니다.
- Unknown의 분포가 특정 클래스군과 가까우면 그쪽으로 예측이 집중될 가능성이 있습니다.

In [ ]:
groups = [
    [1],
    [2],
    [3],
    [4],
    [5],
    [1, 2],
    [3, 4, 5],
    [1, 2, 3, 4, 5]
]

all_x = np.concatenate([features_known_df_lda[:, 0], features_unknown_df_lda[:, 0]])
all_y = np.concatenate([features_known_df_lda[:, 1], features_unknown_df_lda[:, 1]])

margin = 0.5
xlim = (all_x.min() - margin, all_x.max() + margin)
ylim = (all_y.min() - margin, all_y.max() + margin)

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for ax, group in zip(axes[:8], groups):
    for c in group:
        mask = income_known_df == c
        ax.scatter(
            features_known_df_lda[mask, 0],
            features_known_df_lda[mask, 1],
            label=f"class {c}",
            alpha=0.7
        )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(f"LDA 분포 - Classes {group}")
    ax.set_xlabel("LD1")
    ax.set_ylabel("LD2")
    ax.legend()

ax = axes[8]
ax.scatter(
    features_unknown_df_lda[:, 0],
    features_unknown_df_lda[:, 1],
    label="Unknowns",
    color="black",
    alpha=0.7
)
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_title("LDA 분포 - Unknown Data")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.legend()

plt.tight_layout()
plt.show()

# 왜 subplot으로 나누어 보는가:
# - 한 장의 그래프에 모든 클래스를 겹치면 중첩이 심해져 패턴을 읽기 어렵기 때문입니다.
# - 실무 리뷰에서는 '어떤 클래스 조합이 비슷한가'를 비교하는 시각이 중요합니다.

## 15. 이진 분류용 타깃 재정의

### 비즈니스 목적
다중 클래스(`income_id` 1~5)를 실무 의사결정에 맞게 저소득/고소득 성격의 이진 타깃으로 재정의합니다.

### 기대 효과
- 모델 학습과 해석이 단순해집니다.
- 운영 적용 시 정책 기준(예: 우선 고객군 선별)에 더 직접적으로 연결할 수 있습니다.

### 분석 포인트
- 현재 기준은 `income_id <= 2`를 0, `income_id >= 3`을 1로 정의합니다.
- 이 기준은 업무 목적에 따라 조정 가능하므로, 문서화가 중요합니다.

In [ ]:
# 왜 다중 클래스를 이진 분류로 바꾸는가:
# - 클래스가 많을수록 경계가 복잡해지고 해석이 어려워질 수 있으므로,
#   실무 활용을 위해 의사결정 가능한 단순 구조로 바꾸는 경우가 많습니다.
y_data = np.where(income_known_df <= 2, 0, 1)

display(pd.Series(y_data, name="binary_income_group").value_counts().sort_index().rename_axis("class").to_frame("count"))

# 배포 및 유지보수 주의:
# - 0과 1의 의미(저소득/고소득)를 명확히 문서화하지 않으면 운영 단계에서 해석 오류가 발생할 수 있습니다.
# - 타깃 정의가 바뀌면 기존 성능 수치는 비교 대상이 될 수 없습니다.

## 16. 학습 / 검증 데이터 분리

### 비즈니스 목적
모델이 본 데이터와 보지 않은 데이터에서 모두 성능을 내는지 확인하기 위해 train/test split을 수행합니다.

### 기대 효과
- 과적합 여부를 확인할 수 있습니다.
- 실제 배포 시 기대 가능한 일반화 성능을 가늠할 수 있습니다.

### 분석 포인트
- 클래스 비율이 한쪽으로 치우칠 수 있으므로 stratify를 반드시 적용합니다.
- random_state를 고정하면 팀원 간 재현 가능한 결과 비교가 가능합니다.

In [ ]:
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(
    features_known_df_scaled,
    y_data,
    stratify=y_data,
    random_state=42,
    test_size=0.2
)

display(pd.DataFrame({
    "dataset": ["train_X", "test_X", "train_y", "test_y"],
    "shape": [train_X.shape, test_X.shape, train_y.shape, test_y.shape]
}))

# 왜 stratify를 사용하는가:
# - 클래스 비율이 train/test에서 크게 달라지면 성능 비교가 왜곡될 수 있기 때문입니다.
# - 특히 이진 분류에서 소수 클래스가 적을 때 stratify 누락은 평가 신뢰도를 떨어뜨립니다.

## 17. 학습 / 검증 분리 결과 리포트

### 비즈니스 목적
학습/검증 세트의 크기와 클래스 분포가 적절히 유지되었는지 표준 형식으로 점검합니다.

### 기대 효과
- split 이후 데이터 품질을 빠르게 검수할 수 있습니다.
- 팀 공통 리포트 포맷으로 재사용 가능합니다.

### 분석 포인트
- Train/Test의 클래스 비율이 유사해야 합니다.
- 비율 차이가 작을수록 분리 전략이 적절하다고 볼 수 있습니다.

In [ ]:
def print_split_report(X_train, X_test, y_train, y_test):
    split_summary_df = pd.DataFrame({
        "item": [
            "X_train shape", "X_test shape",
            "y_train shape", "y_test shape"
        ],
        "value": [
            str(X_train.shape), str(X_test.shape),
            str(y_train.shape), str(y_test.shape)
        ]
    })

    train_dist = pd.Series(y_train).value_counts().sort_index().rename("train_count")
    test_dist = pd.Series(y_test).value_counts().sort_index().rename("test_count")
    train_ratio = pd.Series(y_train).value_counts(normalize=True).sort_index().rename("train_ratio")
    test_ratio = pd.Series(y_test).value_counts(normalize=True).sort_index().rename("test_ratio")

    class_summary_df = pd.concat([train_dist, test_dist, train_ratio, test_ratio], axis=1)
    display(split_summary_df)
    display(class_summary_df)

print_split_report(train_X, test_X, train_y, test_y)

# 왜 분리 결과를 별도 리포트로 남기는가:
# - 모델 성능 수치를 보기 전에 데이터 분리가 정상인지 먼저 확인해야, 결과를 신뢰할 수 있기 때문입니다.
# - 재학습 시 분포가 달라졌는지 추적하는 기준점 역할도 합니다.

## 18. 모델 평가 함수 정의

### 비즈니스 목적
여러 모델의 성능을 동일한 기준으로 비교하기 위해 공통 평가 함수를 정의합니다.

### 기대 효과
- Accuracy만 보지 않고 Recall, F1, ROC-AUC, PR-AUC까지 함께 확인할 수 있습니다.
- 모델 교체 시에도 동일한 리포팅 형식을 유지할 수 있습니다.

### 분석 포인트
- 클래스 불균형 가능성이 있을 때 PR-AUC와 Recall은 특히 중요합니다.
- 평가 함수는 재사용성이 높아 실무 프로젝트의 표준 자산이 됩니다.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.multiclass import type_of_target

def evaluate_model(y_true, y_pred, y_proba, classes=None, average="macro", dataset_name="Dataset"):
    target_type = type_of_target(y_true)

    result = {
        "dataset": dataset_name,
        "target_type": target_type,
        "accuracy": accuracy_score(y_true, y_pred),
    }

    y_proba = np.asarray(y_proba)

    if target_type == "binary":
        result["recall"] = recall_score(y_true, y_pred)
        result["f1"] = f1_score(y_true, y_pred)

        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            y_score = y_proba[:, 1]
        elif y_proba.ndim == 1:
            y_score = y_proba
        else:
            raise ValueError(f"Binary Classification Shape Error: y_proba.shape = {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(y_true, y_score)
        result["pr_auc"] = average_precision_score(y_true, y_score)
        result["precision"] = precision_score(y_true, y_pred)

    elif target_type == "multiclass":
        result["recall"] = recall_score(y_true, y_pred, average=average)
        result["f1"] = f1_score(y_true, y_pred, average=average)

        if y_proba.ndim != 2:
            raise ValueError(f"Multiclass classification에서는 y_proba가 2차원이어야 합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average=average
        )

        if classes is None:
            classes = np.unique(y_true)

        y_true_bin = label_binarize(y_true, classes=classes)
        result["pr_auc"] = average_precision_score(
            y_true_bin,
            y_proba,
            average=average
        )
        result["precision"] = precision_score(y_true, y_pred, average=average)
    else:
        raise ValueError(f"지원하지 않는 target type입니다: {target_type}")

    return pd.DataFrame([result])

# 왜 공통 평가 함수를 만드는가:
# - 모델마다 출력 형식이 다르면 비교가 어렵고, 지표 계산 로직이 중복되기 때문입니다.
# - 특히 이진/다중 분류를 모두 지원하도록 설계해 두면 다른 프로젝트에도 재사용 가능합니다.

## 19. Logistic Regression 학습 및 평가

### 비즈니스 목적
기준선 모델(Baseline)로서 해석 가능성이 높은 Logistic Regression을 학습하여 기본 성능을 확보합니다.

### 기대 효과
- 복잡한 모델을 쓰기 전에 단순 모델의 분류력을 확인할 수 있습니다.
- 계수 기반 해석이 가능해 설명력이 좋습니다.

### 분석 포인트
- 선형 경계로도 충분한 성능이 나온다면 복잡한 모델이 꼭 필요하지 않을 수 있습니다.
- 테스트 성능이 학습 성능보다 지나치게 낮지 않은지 확인해야 합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(train_X, train_y)

lr_train_pred = lr.predict(train_X)
lr_test_pred = lr.predict(test_X)

lr_train_proba = lr.predict_proba(train_X)
lr_test_proba = lr.predict_proba(test_X)

lr_train_result = evaluate_model(
    y_true=train_y,
    y_pred=lr_train_pred,
    y_proba=lr_train_proba,
    classes=lr.classes_,
    dataset_name="LogisticRegression_Train"
)

lr_test_result = evaluate_model(
    y_true=test_y,
    y_pred=lr_test_pred,
    y_proba=lr_test_proba,
    classes=lr.classes_,
    dataset_name="LogisticRegression_Test"
)

display(pd.concat([lr_train_result, lr_test_result], ignore_index=True))

# 왜 Logistic Regression을 먼저 쓰는가:
# - 구조가 단순하고 빠르며, 결과 해석이 쉬워 기준선 모델로 적합하기 때문입니다.
# - 표준화된 입력과 결합하면 선형 분리 가능성을 빠르게 점검할 수 있습니다.

## 20. Logistic Regression 계수 해석

### 비즈니스 목적
어떤 변수가 고소득 그룹(1) 예측에 상대적으로 영향을 주는지 방향성과 크기를 확인합니다.

### 기대 효과
- 모델 결과를 비즈니스 언어로 설명하기 쉬워집니다.
- 운영 정책과 연결할 수 있는 핵심 변수 후보를 도출할 수 있습니다.

### 분석 포인트
- 양수 계수는 클래스 1 방향, 음수 계수는 클래스 0 방향 기여로 해석할 수 있습니다.
- 절대값이 큰 계수는 상대적으로 영향력이 큰 변수로 볼 수 있습니다.

In [ ]:
lr_coef_df = pd.DataFrame({
    "feature": features_known_df.columns,
    "coefficient": lr.coef_[0],
    "abs_coefficient": np.abs(lr.coef_[0])
}).sort_values("abs_coefficient", ascending=False)

display(lr_coef_df)

plt.figure(figsize=(10, 6))
plt.barh(lr_coef_df["feature"], lr_coef_df["coefficient"])
plt.title("Logistic Regression 변수 계수")
plt.xlabel("계수 값")
plt.ylabel("변수")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 왜 계수를 확인하는가:
# - 단순히 성능만 보는 것이 아니라, 어떤 변수들이 예측에 기여했는지 설명 가능성을 확보해야 하기 때문입니다.
# - 다만 상관이 높은 변수끼리는 계수 해석이 흔들릴 수 있으므로 절대적 인과로 해석하면 안 됩니다.

## 21. RandomForestClassifier 학습 및 평가

### 비즈니스 목적
비선형 패턴까지 반영할 수 있는 트리 기반 모델을 적용해 Logistic Regression과 성능을 비교합니다.

### 기대 효과
- 변수 간 복잡한 상호작용을 반영할 수 있습니다.
- 선형 모델 대비 성능 개선 여부를 점검할 수 있습니다.

### 분석 포인트
- 트리 모델은 스케일링 민감도가 낮지만, 이번 비교에서는 동일 입력을 사용합니다.
- 학습 성능이 지나치게 높고 테스트 성능과 차이가 크면 과적합 가능성을 의심해야 합니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    random_state=42,
    max_depth=3,
    n_estimators=200
)
rf_clf.fit(train_X, train_y)

rf_train_pred = rf_clf.predict(train_X)
rf_test_pred = rf_clf.predict(test_X)

rf_train_proba = rf_clf.predict_proba(train_X)
rf_test_proba = rf_clf.predict_proba(test_X)

rf_train_result = evaluate_model(
    y_true=train_y,
    y_pred=rf_train_pred,
    y_proba=rf_train_proba,
    classes=rf_clf.classes_,
    dataset_name="RandomForest_Train"
)

rf_test_result = evaluate_model(
    y_true=test_y,
    y_pred=rf_test_pred,
    y_proba=rf_test_proba,
    classes=rf_clf.classes_,
    dataset_name="RandomForest_Test"
)

display(pd.concat([rf_train_result, rf_test_result], ignore_index=True))

# 왜 별도 변수명(train_proba / test_proba)을 명확히 나누는가:
# - 원본 코드처럼 변수명이 섞이면 다른 모델의 확률값을 잘못 참조하는 버그가 발생할 수 있기 때문입니다.
# - 실무에서는 모델별 산출물을 접두어(lr_, rf_)로 관리하는 것이 안전합니다.

## 22. RandomForest 변수 중요도 해석

### 비즈니스 목적
트리 기반 모델이 어떤 변수를 중심으로 분기했는지 확인하여 변수 중요도를 검토합니다.

### 기대 효과
- 비선형 모델 관점의 핵심 변수 후보를 파악할 수 있습니다.
- 선형 모델 계수와 다른 관점의 설명 자료를 확보할 수 있습니다.

### 분석 포인트
- 중요도가 높다고 해서 방향성까지 알 수 있는 것은 아닙니다.
- 선형 계수와 트리 중요도를 함께 보면 공통 핵심 변수와 모델별 차이를 비교할 수 있습니다.

In [ ]:
rf_importance_df = pd.DataFrame({
    "feature": features_known_df.columns,
    "importance": rf_clf.feature_importances_
}).sort_values("importance", ascending=False)

display(rf_importance_df)

plt.figure(figsize=(10, 6))
plt.barh(rf_importance_df["feature"], rf_importance_df["importance"])
plt.title("RandomForest 변수 중요도")
plt.xlabel("중요도")
plt.ylabel("변수")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 왜 변수 중요도를 보는가:
# - 트리 모델은 계수 해석이 어렵기 때문에, 어떤 변수가 분기에 많이 활용되었는지를 별도로 확인해야 하기 때문입니다.
# - 다만 중요도는 분산이 큰 변수에 유리할 수 있어, 절대적 해석보다 참고 지표로 쓰는 것이 바람직합니다.

## 23. 모델 성능 비교표 정리

### 비즈니스 목적
두 모델의 Train/Test 성능을 한 표로 정리해 최종 후보 모델을 선택합니다.

### 기대 효과
- 비교 기준이 한눈에 정리되어 의사결정이 쉬워집니다.
- 모델 선택 근거를 문서로 남길 수 있습니다.

### 분석 포인트
- 일반적으로 Test 성능을 우선적으로 보고, Train/Test 차이로 과적합 여부를 함께 판단합니다.
- 운영 목적에 따라 Accuracy보다 Recall, PR-AUC를 더 중요하게 볼 수 있습니다.

In [ ]:
model_compare_df = pd.concat(
    [lr_train_result, lr_test_result, rf_train_result, rf_test_result],
    ignore_index=True
)

display(model_compare_df)

# 왜 비교표를 마지막에 다시 모으는가:
# - 셀별로 흩어진 결과는 회의나 리뷰에서 비교가 어렵기 때문입니다.
# - 모델 선택 근거를 명확히 남기려면 동일 포맷의 단일 표가 필요합니다.

## 24. Unknown 그룹 예측 수행

### 비즈니스 목적
학습된 두 모델을 Unknown 소득군에 적용해, 실제 운영 시 어떤 분류 결과가 나오는지 확인합니다.

### 기대 효과
- 미확인 고객군을 임시 분류하여 후속 업무(우선 검토, 캠페인 분기 등)에 활용할 수 있습니다.
- 모델 간 합의 정도를 통해 예측 신뢰도를 점검할 수 있습니다.

### 분석 포인트
- Unknown 예측은 정답이 없는 추론 결과이므로, 성능 평가가 아니라 운영 보조 정보로 해석해야 합니다.
- 모델 간 예측이 일치하는 구간은 상대적으로 신뢰도가 높을 수 있습니다.

In [ ]:
def print_valcnt(arr):
    values, counts = np.unique(arr, return_counts=True)
    result_df = pd.DataFrame({"predicted_class": values, "count": counts})
    display(result_df)

lr_predicted_unknown = lr.predict(features_unknown_df_scaled)
rf_predicted_unknown = rf_clf.predict(features_unknown_df_scaled)

print("Logistic Regression 예측 분포")
print_valcnt(lr_predicted_unknown)

print("Random Forest 예측 분포")
print_valcnt(rf_predicted_unknown)

# 왜 Unknown 예측 분포를 먼저 보는가:
# - 한쪽 클래스로 지나치게 몰리는지 확인해야, 운영 적용 시 편향 위험을 판단할 수 있기 때문입니다.
# - 정답이 없는 데이터이므로 개별 정확도보다 전체 분포와 패턴이 우선 중요합니다.

## 25. Unknown 그룹 예측 불일치 분석

### 비즈니스 목적
두 모델이 서로 다르게 판단한 미확인 고객군을 별도로 추출하여, 추가 검토가 필요한 위험 구간을 찾습니다.

### 기대 효과
- 운영 검토 우선순위를 정할 수 있습니다.
- 후속 라벨링 또는 도메인 검토 대상을 효율적으로 선정할 수 있습니다.

### 분석 포인트
- 모델 불일치 건수는 불확실성의 간단한 대리 지표로 볼 수 있습니다.
- 불일치 샘플의 공통 특징을 추가 분석하면 규칙 개선에 도움이 됩니다.

In [ ]:
disagree_mask = lr_predicted_unknown != rf_predicted_unknown

unknown_prediction_compare_df = pd.DataFrame({
    "lr_pred": lr_predicted_unknown,
    "rf_pred": rf_predicted_unknown
}, index=features_unknown_df.index)

disagree_df = unknown_prediction_compare_df[disagree_mask].copy()

display(pd.DataFrame({
    "unknown_total_count": [len(unknown_prediction_compare_df)],
    "disagree_count": [disagree_mask.sum()],
    "disagree_ratio": [disagree_mask.mean()]
}))

display(disagree_df.head(20))

# 왜 불일치 데이터를 따로 보는가:
# - 여러 모델이 동일하게 예측한 샘플보다, 다르게 예측한 샘플이 운영 리스크가 더 크기 때문입니다.
# - 이 구간은 추가 라벨링, 수동 검토, 규칙 기반 보완의 우선 대상이 됩니다.

## 26. 불일치 샘플 원본 피처 확인

### 비즈니스 목적
모델 간 예측이 엇갈린 Unknown 고객의 원본 피처를 직접 확인해, 어떤 특성에서 애매한 경계가 발생하는지 점검합니다.

### 기대 효과
- 운영/도메인 담당자와 함께 검토할 수 있는 구체적 사례를 확보할 수 있습니다.
- 모델 개선 또는 타깃 재정의 논의의 근거 자료가 됩니다.

### 분석 포인트
- 경계 구간 샘플은 여러 변수에서 중간값 근처를 보일 가능성이 있습니다.
- 향후 확률 기반 임계값 정책을 도입할 때 참고할 수 있습니다.

In [ ]:
if len(disagree_df) > 0:
    disagree_feature_df = features_unknown_df.loc[disagree_df.index].copy()
    disagree_feature_df["lr_pred"] = disagree_df["lr_pred"]
    disagree_feature_df["rf_pred"] = disagree_df["rf_pred"]
    display(disagree_feature_df.head(20))
else:
    print("두 모델 간 예측 불일치 샘플이 없습니다.")

# 왜 원본 피처를 다시 붙여보는가:
# - 예측 결과 숫자만 보면 실제 어떤 고객 특성에서 판단이 갈렸는지 알 수 없기 때문입니다.
# - 실무에서는 모델 결과보다 '어떤 고객을 왜 다르게 봤는가'가 더 중요한 설명 포인트가 됩니다.

## 27. 결과 저장

### 비즈니스 목적
핵심 산출물(모델 비교표, Unknown 예측 결과)을 파일로 저장해 재사용 가능하게 만듭니다.

### 기대 효과
- 후속 보고서, 대시보드, 검토 문서에 쉽게 연결할 수 있습니다.
- 노트북 재실행 없이도 결과 파일을 공유할 수 있습니다.

### 분석 포인트
- 저장 경로를 명시적으로 관리해야 협업 환경에서 파일 누락을 줄일 수 있습니다.
- 파일명에 분석 주제를 포함하면 버전 관리가 쉬워집니다.

In [ ]:
import os

output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

model_compare_path = os.path.join(output_dir, "model_compare_income_binary.csv")
unknown_pred_path = os.path.join(output_dir, "unknown_income_predictions.csv")

model_compare_df.to_csv(model_compare_path, index=False, encoding="utf-8-sig")
unknown_prediction_compare_df.to_csv(unknown_pred_path, index=True, encoding="utf-8-sig")

display(pd.DataFrame({
    "saved_file": [model_compare_path, unknown_pred_path]
}))

# 왜 CSV로 저장하는가:
# - 노트북을 열지 않아도 엑셀, BI 도구, 다른 스크립트에서 쉽게 재사용할 수 있기 때문입니다.
# 배포 및 유지보수 주의:
# - 상대경로는 실행 위치에 따라 달라질 수 있으므로, 운영 배치에서는 절대경로 또는 환경변수 기반 경로 관리가 더 안전합니다.
# - 인코딩 이슈를 줄이기 위해 utf-8-sig를 사용했습니다.

## 28. 최종 결론

### 비즈니스 관점 요약
이번 분석은 `income_id == 6`으로 표시된 미확인 소득군을 분리한 뒤, 알려진 소득군 패턴만으로 소득 구간을 추정할 수 있는지 검토한 작업입니다.  
LDA 시각화와 변수 분포 비교를 통해 Known/Unknown 간 차이를 확인했고, 실무 적용을 위해 다중 클래스 대신 이진 분류 기준으로 문제를 재정의했습니다.

### 실무 해석
- 선형 모델(Logistic Regression)과 비선형 모델(Random Forest)을 함께 비교함으로써, 단순성과 성능 사이의 균형을 검토할 수 있습니다.
- Unknown 그룹에 대한 모델 예측 분포와 불일치 샘플을 별도로 관리하면, 운영 적용 시 검토 우선순위를 세우기 좋습니다.
- 모델 불일치 데이터는 곧바로 자동 처리하기보다 추가 검증 대상으로 두는 것이 안전합니다.

### 후속 제안
1. Unknown 그룹에 대한 실제 라벨 확보가 가능하다면, 별도 검증셋으로 모델 신뢰도를 확인하는 것이 가장 중요합니다.
2. 현재의 이진 기준(`income_id <= 2` vs `>= 3`)이 업무 목적에 최적인지 도메인 관점 검토가 필요합니다.
3. 확률값 기반 임계값 조정, 교차검증, 클래스 불균형 대응(class_weight 등)을 추가 적용하면 운영 안정성을 더 높일 수 있습니다.